In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch

df = pd.read_csv("../data/synthetic/text/corpus.csv")
print(df.shape)
print(df["label"].value_counts())
print(df["language"].value_counts())

(800, 4)
label
phishing    400
genuine     400
Name: count, dtype: int64
language
English    400
Hindi      400
Name: count, dtype: int64


In [2]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])
print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df[["label", "language"]], random_state=42
)

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")
print(train_df["label"].value_counts())
print(test_df["label"].value_counts())

{'genuine': np.int64(0), 'phishing': np.int64(1)}
Train size: 640, Test size: 160
label
genuine     320
phishing    320
Name: count, dtype: int64
label
phishing    80
genuine     80
Name: count, dtype: int64


In [3]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_pandas(train_df[["text", "label_id"]].rename(columns={"label_id": "label"}))
test_dataset = Dataset.from_pandas(test_df[["text", "label_id"]].rename(columns={"label_id": "label"}))

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print(train_dataset)
print(test_dataset[0].keys())

Map:   0%|          | 0/640 [00:00<?, ? examples/s]

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 640
})
dict_keys(['text', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'])


In [4]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Using device: {device}")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

training_args = TrainingArguments(
    output_dir="../models/detection/mbert_checkpoints",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="../models/detection/logs",
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transforme

Using device: cuda


In [5]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.034889,0.008677,1.000000,1.000000,1.000000,1.000000
2,0.000366,0.000242,1.000000,1.000000,1.000000,1.000000
3,0.000236,0.000166,1.000000,1.000000,1.000000,1.000000
4,0.000203,0.000151,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=160, training_loss=0.06746099974843674, metrics={'train_runtime': 459.88, 'train_samples_per_second': 5.567, 'train_steps_per_second': 0.348, 'total_flos': 336782150860800.0, 'train_loss': 0.06746099974843674, 'epoch': 4.0})

In [6]:
eval_results = trainer.evaluate()
print(eval_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.000203,0.008551,4,1.000000,1.000000,1.000000,1.000000


{'eval_loss': 0.008551336824893951, 'eval_accuracy': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_f1': 1.0}


In [7]:
import re

def get_template_signature(text):
    return re.sub(r'\d+|₹[\d,]+|SEBI/[A-Z]+/\d+/[A-Z]*\d+', 'X', text)[:50]

df["template_sig"] = df["text"].apply(get_template_signature)
unique_sigs = df["template_sig"].unique().tolist()
print(f"Unique template signatures: {len(unique_sigs)}")

train_sigs, test_sigs = train_test_split(unique_sigs, test_size=0.25, random_state=42)

train_df2 = df[df["template_sig"].isin(train_sigs)]
test_df2 = df[df["template_sig"].isin(test_sigs)]

print(f"Train: {len(train_df2)}, Test: {len(test_df2)}")
print(train_df2["label"].value_counts())
print(test_df2["label"].value_counts())

Unique template signatures: 336
Train: 609, Test: 191
label
phishing    313
genuine     296
Name: count, dtype: int64
label
genuine     104
phishing     87
Name: count, dtype: int64


In [8]:
train_dataset2 = Dataset.from_pandas(train_df2[["text", "label_id"]].rename(columns={"label_id": "label"}))
test_dataset2 = Dataset.from_pandas(test_df2[["text", "label_id"]].rename(columns={"label_id": "label"}))

train_dataset2 = train_dataset2.map(tokenize_function, batched=True)
test_dataset2 = test_dataset2.map(tokenize_function, batched=True)

model2 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model2.to(device)

training_args2 = TrainingArguments(
    output_dir="../models/detection/mbert_checkpoints_v2",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="../models/detection/logs_v2",
    logging_steps=10,
    report_to="none"
)

trainer2 = Trainer(
    model=model2,
    args=training_args2,
    train_dataset=train_dataset2,
    eval_dataset=test_dataset2,
    compute_metrics=compute_metrics
)

trainer2.train()

Map:   0%|          | 0/609 [00:00<?, ? examples/s]

Map:   0%|          | 0/191 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transforme

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.031101,0.000705,1.000000,1.000000,1.000000,1.000000
2,0.000379,0.000230,1.000000,1.000000,1.000000,1.000000
3,0.000267,0.000178,1.000000,1.000000,1.000000,1.000000
4,0.000233,0.000165,1.000000,1.000000,1.000000,1.000000


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=156, training_loss=0.03949694922611786, metrics={'train_runtime': 1190.8472, 'train_samples_per_second': 2.046, 'train_steps_per_second': 0.131, 'total_flos': 320469265428480.0, 'train_loss': 0.03949694922611786, 'epoch': 4.0})

In [14]:
test_examples = [
    ("Your investment matured today. Please log in to check your updated portfolio value and download your statement.", "genuine"),
    ("Hi, myself Rakesh from stock market department, your file has been selected for special bonus, please share your PAN and bank details to release fund", "phishing"),
    ("As per the notification issued this week, the settlement cycle for the derivative segment has been revised effective next month.", "genuine"),
    ("अगर आप 1 लाख अभी लगाते हैं तो 15 दिन में 3 लाख वापस मिलेगा, बस अपना OTP भेज दीजिये सर", "phishing"),
    ("आपका फोलियो स्टेटमेंट तैयार है, कृपया अपने पंजीकृत ईमेल पर जांचें।", "genuine"),
]

model3.eval()
for text, true_label in test_examples:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model3(**inputs)
        pred = torch.argmax(outputs.logits, dim=-1).item()
        pred_label = label_encoder.inverse_transform([pred])[0]
    print(f"True: {true_label:10s} | Predicted: {pred_label:10s} | Text: {text[:60]}...")

NameError: name 'model3' is not defined

In [ ]:
df = pd.read_csv("../data/synthetic/text/corpus.csv")
print(df.shape)
print(df.columns.tolist())
print(df["template_id"].nunique())
df.head()

(800, 4)
['text', 'label', 'language', 'template_id']
48


,text,label,language,template_id
0,Congratulations Rajesh Kumar! You have been se...,phishing,English,en_phishing_1
1,Congratulations Manoj Verma! You have been sel...,phishing,English,en_phishing_1
2,Congratulations Sneha Reddy! You have been sel...,phishing,English,en_phishing_1
3,Insider tip: HDFC Top 100 Fund is about to ann...,phishing,English,en_phishing_3
4,URGENT: SEBI Circular No. SEBI/MIRSD/2026/4521...,phishing,English,en_phishing_0


In [11]:
template_labels = df.drop_duplicates("template_id")[["template_id", "label"]]

train_ids, test_ids = train_test_split(
    template_labels["template_id"],
    test_size=0.25,
    random_state=42,
    stratify=template_labels["label"]
)

train_df3 = df[df["template_id"].isin(train_ids)]
test_df3 = df[df["template_id"].isin(test_ids)]

print(f"Train templates: {len(train_ids)}, Test templates: {len(test_ids)}")
print(f"Train rows: {len(train_df3)}, Test rows: {len(test_df3)}")
print("Train label distribution:")
print(train_df3["label"].value_counts())
print("Test label distribution:")
print(test_df3["label"].value_counts())

Train templates: 39, Test templates: 13
Train rows: 548, Test rows: 252
Train label distribution:
label
phishing    314
genuine     234
Name: count, dtype: int64
Test label distribution:
label
genuine     166
phishing     86
Name: count, dtype: int64


In [12]:
train_encodings3 = tokenizer(
    train_df3["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)
test_encodings3 = tokenizer(
    test_df3["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

train_labels3 = label_encoder.transform(train_df3["label"]).tolist()
test_labels3 = label_encoder.transform(test_df3["label"]).tolist()

class PhishingDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset3 = PhishingDataset(train_encodings3, train_labels3)
test_dataset3 = PhishingDataset(test_encodings3, test_labels3)

print(f"Train dataset size: {len(train_dataset3)}")
print(f"Test dataset size: {len(test_dataset3)}")

Train dataset size: 548
Test dataset size: 252


In [ ]:
model3 = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=2
).to(device)

training_args3 = TrainingArguments(
    output_dir="../models/detection/mbert_checkpoints_v3",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="../models/detection/logs_v3",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

trainer3 = Trainer(
    model=model3,
    args=training_args3,
    train_dataset=train_dataset3,
    eval_dataset=test_dataset3,
    compute_metrics=compute_metrics
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transforme

In [13]:
trainer3.train()

NameError: name 'trainer3' is not defined

In [ ]:
import numpy as np

predictions_output = trainer3.predict(test_dataset3)
preds = np.argmax(predictions_output.predictions, axis=1)

pred_labels = label_encoder.inverse_transform(preds)
true_labels = label_encoder.inverse_transform(test_labels3)

import pandas as pd
results_df = pd.DataFrame({
    "text": test_df3["text"].tolist(),
    "true": true_labels,
    "pred": pred_labels
})

print(results_df["pred"].value_counts())
print()
print("Misclassified phishing (predicted genuine):")
print(results_df[(results_df["true"] == "phishing") & (results_df["pred"] == "genuine")]["text"].tolist())

pred
genuine     150
phishing     74
Name: count, dtype: int64

Misclassified phishing (predicted genuine):
['आपके Sharekhan ट्रेडिंग खाते में ₹25,000 मूल्य का अनक्लेम्ड IPO आवंटन दिखाया गया है। अपने शेयर दावा करने के लिए लिंक पर क्लिक करें और अपना नेट बैंकिंग पासवर्ड दर्ज करके सत्यापन पूरा करें।', 'आपके Upstox ट्रेडिंग खाते में ₹75,000 मूल्य का अनक्लेम्ड IPO आवंटन दिखाया गया है। अपने शेयर दावा करने के लिए लिंक पर क्लिक करें और अपना नेट बैंकिंग पासवर्ड दर्ज करके सत्यापन पूरा करें।', 'आपके Angel One ट्रेडिंग खाते में ₹50,000 मूल्य का अनक्लेम्ड IPO आवंटन दिखाया गया है। अपने शेयर दावा करने के लिए लिंक पर क्लिक करें और अपना नेट बैंकिंग पासवर्ड दर्ज करके सत्यापन पूरा करें।', 'आपके HDFC Securities ट्रेडिंग खाते में ₹25,000 मूल्य का अनक्लेम्ड IPO आवंटन दिखाया गया है। अपने शेयर दावा करने के लिए लिंक पर क्लिक करें और अपना नेट बैंकिंग पासवर्ड दर्ज करके सत्यापन पूरा करें।', 'आपके Upstox ट्रेडिंग खाते में ₹5,00,000 मूल्य का अनक्लेम्ड IPO आवंटन दिखाया गया है। अपने शेयर दावा करने के लिए लिंक पर क्लि

In [ ]:
missed = results_df[(results_df["true"] == "phishing") & (results_df["pred"] == "genuine")]
print(f"Total missed: {len(missed)}")
for t in missed["text"]:
    print("-", t)
    print()

Total missed: 24
- आपके Sharekhan ट्रेडिंग खाते में ₹25,000 मूल्य का अनक्लेम्ड IPO आवंटन दिखाया गया है। अपने शेयर दावा करने के लिए लिंक पर क्लिक करें और अपना नेट बैंकिंग पासवर्ड दर्ज करके सत्यापन पूरा करें।

- आपके Upstox ट्रेडिंग खाते में ₹75,000 मूल्य का अनक्लेम्ड IPO आवंटन दिखाया गया है। अपने शेयर दावा करने के लिए लिंक पर क्लिक करें और अपना नेट बैंकिंग पासवर्ड दर्ज करके सत्यापन पूरा करें।

- आपके Angel One ट्रेडिंग खाते में ₹50,000 मूल्य का अनक्लेम्ड IPO आवंटन दिखाया गया है। अपने शेयर दावा करने के लिए लिंक पर क्लिक करें और अपना नेट बैंकिंग पासवर्ड दर्ज करके सत्यापन पूरा करें।

- आपके HDFC Securities ट्रेडिंग खाते में ₹25,000 मूल्य का अनक्लेम्ड IPO आवंटन दिखाया गया है। अपने शेयर दावा करने के लिए लिंक पर क्लिक करें और अपना नेट बैंकिंग पासवर्ड दर्ज करके सत्यापन पूरा करें।

- आपके Upstox ट्रेडिंग खाते में ₹5,00,000 मूल्य का अनक्लेम्ड IPO आवंटन दिखाया गया है। अपने शेयर दावा करने के लिए लिंक पर क्लिक करें और अपना नेट बैंकिंग पासवर्ड दर्ज करके सत्यापन पूरा करें।

- आपके HDFC Securities ट्र

In [ ]:
false_positives = results_df[(results_df["true"] == "genuine") & (results_df["pred"] == "phishing")]
print(f"Total false positives: {len(false_positives)}")
for t in false_positives["text"]:
    print("-", t)

Total false positives: 0


In [ ]:
hindi_phishing_test = results_df.merge(
    test_df3[["text", "template_id"]], on="text", how="left"
)
print(hindi_phishing_test[(hindi_phishing_test["true"] == "phishing")].groupby("template_id")["pred"].value_counts())

template_id    pred    
en_phishing_0  phishing    20
hi_phishing_1  phishing    31
hi_phishing_6  genuine     24
hi_phishing_7  phishing    23
Name: count, dtype: int64


In [ ]:
df = pd.read_csv("../data/synthetic/text/corpus.csv")
print(df.shape)
print(df["template_id"].nunique())

template_labels = df.drop_duplicates("template_id")[["template_id", "label"]]

train_ids4, test_ids4 = train_test_split(
    template_labels["template_id"],
    test_size=0.25,
    random_state=42,
    stratify=template_labels["label"]
)

train_df4 = df[df["template_id"].isin(train_ids4)]
test_df4 = df[df["template_id"].isin(test_ids4)]

print(f"Train templates: {len(train_ids4)}, Test templates: {len(test_ids4)}")
print(f"Train rows: {len(train_df4)}, Test rows: {len(test_df4)}")
print("Train label distribution:")
print(train_df4["label"].value_counts())
print("Test label distribution:")
print(test_df4["label"].value_counts())

(800, 4)
52
Train templates: 39, Test templates: 13
Train rows: 548, Test rows: 252
Train label distribution:
label
phishing    314
genuine     234
Name: count, dtype: int64
Test label distribution:
label
genuine     166
phishing     86
Name: count, dtype: int64


In [ ]:
train_encodings4 = tokenizer(
    train_df4["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)
test_encodings4 = tokenizer(
    test_df4["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

train_labels4 = label_encoder.transform(train_df4["label"]).tolist()
test_labels4 = label_encoder.transform(test_df4["label"]).tolist()

train_dataset4 = PhishingDataset(train_encodings4, train_labels4)
test_dataset4 = PhishingDataset(test_encodings4, test_labels4)

print(f"Train dataset size: {len(train_dataset4)}")
print(f"Test dataset size: {len(test_dataset4)}")

model4 = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=2
).to(device)

training_args4 = TrainingArguments(
    output_dir="../models/detection/mbert_checkpoints_v4",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="../models/detection/logs_v4",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

trainer4 = Trainer(
    model=model4,
    args=training_args4,
    train_dataset=train_dataset4,
    eval_dataset=test_dataset4,
    compute_metrics=compute_metrics
)

NameError: name 'PhishingDataset' is not defined

In [ ]:
trainer4.train()

NameError: name 'trainer4' is not defined

In [ ]:
import gc
import torch

for var_name in ["model", "model2", "model3", "trainer", "trainer2", "trainer3"]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
print(torch.cuda.memory_reserved() / 1e9, "GB reserved")

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
print(torch.cuda.memory_reserved() / 1e9, "GB reserved")

True
0.0 GB allocated
0.0 GB reserved


In [15]:
model3 = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=2
).to(device)

training_args3 = TrainingArguments(
    output_dir="../models/detection/mbert_checkpoints_v3",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="../models/detection/logs_v3",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

trainer3 = Trainer(
    model=model3,
    args=training_args3,
    train_dataset=train_dataset3,
    eval_dataset=test_dataset3,
    compute_metrics=compute_metrics
)

trainer3.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transforme

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.052872,0.029430,1.000000,1.000000,1.000000,1.000000
2,0.000603,2.100805,0.535714,0.423645,1.000000,0.595156
3,0.000393,1.498099,0.615079,0.469945,1.000000,0.639405
4,0.000320,1.320984,0.634921,0.483146,1.000000,0.651515


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=140, training_loss=0.06658927982207388, metrics={'train_runtime': 1000.8419, 'train_samples_per_second': 2.19, 'train_steps_per_second': 0.14, 'total_flos': 138552637308480.0, 'train_loss': 0.06658927982207388, 'epoch': 4.0})

In [16]:
print(trainer3.state.best_model_checkpoint)
print(trainer3.state.best_metric)

../models/detection/mbert_checkpoints_v3\checkpoint-35
1.0


In [17]:
model3.eval()
for text, true_label in test_examples:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model3(**inputs)
        pred = torch.argmax(outputs.logits, dim=-1).item()
        pred_label = label_encoder.inverse_transform([pred])[0]
    print(f"True: {true_label:10s} | Predicted: {pred_label:10s} | Text: {text[:60]}...")

True: genuine    | Predicted: genuine    | Text: Your investment matured today. Please log in to check your u...
True: phishing   | Predicted: genuine    | Text: Hi, myself Rakesh from stock market department, your file ha...
True: genuine    | Predicted: genuine    | Text: As per the notification issued this week, the settlement cyc...
True: phishing   | Predicted: phishing   | Text: अगर आप 1 लाख अभी लगाते हैं तो 15 दिन में 3 लाख वापस मिलेगा, ...
True: genuine    | Predicted: genuine    | Text: आपका फोलियो स्टेटमेंट तैयार है, कृपया अपने पंजीकृत ईमेल पर ज...


In [18]:
model3.save_pretrained("../models/detection/mbert_final")
tokenizer.save_pretrained("../models/detection/mbert_final")
print("Model saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved.


In [19]:
print(df["template_id"].nunique())

52


In [20]:
import os
print(os.path.exists("../models/detection/mbert_final"))
print(os.listdir("../models/detection/mbert_final"))

True
['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']


In [21]:
import pandas as pd

real_df = pd.read_csv("../data/synthetic/text/genuine_real_excerpts.csv")
print(real_df.shape)
print(real_df.columns.tolist())
real_df.head()

(250, 5)
['text', 'label', 'language', 'source', 'template_id']


,text,label,language,source,template_id
0,As an on-going measure to enhance ease of deal...,genuine,English,real_circular,real_circular_0
1,"SEBI, vide Circular No. SEBI/HO/IMD/IMD-I POD-...",genuine,English,real_circular,real_circular_1
2,"SEBI, vide notification dated July 01, 2026, h...",genuine,English,real_circular,real_circular_2
3,Mutual Fund investors can avail the facility o...,genuine,English,real_circular,real_circular_3
4,In order to address liquidity mismatches due t...,genuine,English,real_circular,real_circular_4


In [22]:
model3.eval()
correct = 0
misclassified = []

for text in real_df["text"]:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model3(**inputs)
        pred = torch.argmax(outputs.logits, dim=-1).item()
        pred_label = label_encoder.inverse_transform([pred])[0]
    if pred_label == "genuine":
        correct += 1
    else:
        misclassified.append(text)

accuracy = correct / len(real_df)
print(f"Correct: {correct} / {len(real_df)}")
print(f"Accuracy on real excerpts: {accuracy:.3f}")
print()
print(f"Misclassified count: {len(misclassified)}")

Correct: 249 / 250
Accuracy on real excerpts: 0.996

Misclassified count: 1


In [23]:
print(misclassified[0])

SEBI consultation paper dated February 24, 2025 on ‘Enhancing Trading Convenience and Strengthening Risk Monitoring in Equity Derivatives’, proposed the following Future Equivalent (FutEq) or delta equivalent positions limits for index options: S No. Position type Limit 1 End of day Net FutEq : ₹500 crores Gross FutEq : ₹1,500 crores 2 Intraday Net FutEq : ₹1,000 crores Gross FutEq : ₹2,500 crores 2.


In [24]:
print(label_encoder.classes_)

['genuine' 'phishing']
